##Transforming sprints data

In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/1.environment_config

In [0]:
%run ../00-common/3.silver_helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"

## 1. Schema Shaping Steps

In [0]:
sprints_df = (
    spark.table(bronze_table)
    .select(
        F.col("season"),
        F.col("round"),
        F.col("constructorId"),
        F.col("driverId"),
        F.col("date"),
        F.col("raceName"),
        F.col("grid"),
        F.col("laps"),
        F.col("number"),
        F.col("points"),
        F.col("position"),
        F.col("positionText"),
        F.col("status"),
        F.col("ingestion_timestamp"),
        F.col("source_file"),
        F.col("batch_id")
    ).withColumnsRenamed(
        {
        "constructorId": "constructor_id",
        "driverId": "driver_id",
        "raceName": "race_name",
        "positionText": "final_position_text",
        "date": "race_date",
        "grid": "grid_position",
        "laps": "completed_laps",
        "number": "car_number",
        "position": "final_position"
        }
    )
)

##2. Data Quality Cheks

In [0]:
sprints_valid_df = (
    sprints_df.filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull() &
        F.col("constructor_id").isNotNull() &
        F.col("driver_id").isNotNull()
    ).dropDuplicates(
        ["season", "round", "constructor_id", "driver_id"]
    )
)


In [0]:
print("number of rows dropped: ", sprints_df.count()-sprints_valid_df.count())

## 3. Column Value Level Transformations

In [0]:
sprints_final_df = (
    sprints_valid_df
    .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
display(sprints_final_df)

## 4. Writing to the Silver Delta Table

In [0]:
sprints_columns_to_update = [
    "season",
    "round",
    "constructor_id",
    "driver_id",
    "race_date",
    "race_name",
    "grid_position",
    "completed_laps",
    "car_number",
    "points",
    "final_position",
    "final_position_text",
    "status",
    "ingestion_timestamp",
    "source_file",
    "batch_id"
]

write_to_silver(
    sprints_final_df, 
    silver_table,
    merge_condition = "t.season = s.season AND t.round = s.round AND t.constructor_id = s.constructor_id AND t.driver_id = s.driver_id",
    columns_to_update = sprints_columns_to_update
    )

In [0]:
spark.table(silver_table).display()